# Nhập thư viện và cấu hình
- Nhập các thư viện cần thiết: `pandas`, `numpy`, `matplotlib` để xử lý và trực quan hóa dữ liệu; `statsmodels` và `pmdarima` để phân tích và dự báo chuỗi thời gian.
- Tắt cảnh báo để tránh thông báo không cần thiết.
- Cấu hình `matplotlib` với style `seaborn-v0_8-darkgrid` để biểu đồ có giao diện đẹp.
- Định nghĩa bảng màu `colors` để sử dụng cho các biểu đồ sau này.
- In thông báo xác nhận đã hoàn tất việc nhập thư viện và cấu hình.

In [105]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from statsmodels.tsa.stattools import acf
import pmdarima as pm
import warnings
warnings.filterwarnings('ignore')

# Cấu hình matplotlib
plt.style.use('seaborn-v0_8-darkgrid')  # nếu dùng matplotlib mới


# Định nghĩa màu sắc cho biểu đồ
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', 
          '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']

print("Thư viện đã được import và cấu hình hoàn tất.")

Thư viện đã được import và cấu hình hoàn tất.


# Đọc và xử lý dữ liệu
- Đọc dữ liệu từ file `order_history_kaggle_data.csv` vào DataFrame `df`.
- Chuyển đổi cột `Order Placed At` thành định dạng `datetime`, báo lỗi nếu có giá trị không hợp lệ.
- Định nghĩa hàm `parse_items` để xử lý cột `Items in order`:
  - Tách tên món và số lượng từ chuỗi (ví dụ: "1 x Grilled Chicken: 2" → ("Grilled Chicken", 2)).
  - Xử lý ngoại lệ nếu chuỗi không đúng định dạng.
- Áp dụng hàm `parse_items` để tạo cột `Parsed Items` chứa danh sách các món đã được phân tích.
- In 5 hàng đầu tiên của dữ liệu (các cột `Order Placed At` và `Parsed Items`) và tổng số hàng để kiểm tra.

In [106]:
# Đọc dữ liệu
try:
    df = pd.read_csv('order_history_kaggle_data.csv')
except FileNotFoundError:
    raise FileNotFoundError("Không tìm thấy file 'order_history_kaggle_data.csv'. Kiểm tra đường dẫn file.")

# Chuyển đổi cột Order Placed At thành datetime
df['Order Placed At'] = pd.to_datetime(df['Order Placed At'], errors='coerce')
if df['Order Placed At'].isna().any():
    raise ValueError("Cột 'Order Placed At' chứa giá trị không hợp lệ. Kiểm tra định dạng ngày tháng.")

# Hàm xử lý cột Items in order
def parse_items(items_str):
    try:
        if pd.isna(items_str):
            return []
        items = []
        for item in items_str.split(','):
            item = item.strip()
            if ':' in item:
                name, qty = item.rsplit(':', 1)
                name = name.strip()
                qty = int(qty.strip())
                items.append((name, qty))
            else:
                items.append((item, 1))
        return items
    except Exception as e:
        print(f"Lỗi khi xử lý items: {items_str}. Lỗi: {e}")
        return []

# Áp dụng parse_items
df['Parsed Items'] = df['Items in order'].apply(parse_items)

# Kiểm tra dữ liệu
print("Dữ liệu đầu tiên (5 hàng):")
print(df[['Order Placed At', 'Parsed Items']].head())
print(f"Tổng số hàng: {len(df)}")

Dữ liệu đầu tiên (5 hàng):
      Order Placed At                                       Parsed Items
0 2024-09-10 23:38:00  [(1 x Grilled Chicken Jamaican Tender, 1), (1 ...
1 2024-09-10 23:34:00  [(1 x Peri Peri Fries, 1), (1 x Fried Chicken ...
2 2024-09-10 15:52:00       [(1 x Bone in Peri Peri Grilled Chicken, 1)]
3 2024-09-10 15:45:00  [(1 x Fried Chicken Ghostbuster Tender, 1), (1...
4 2024-09-10 15:04:00  [(1 x Peri Peri Krispers, 1), (1 x Fried Chick...
Tổng số hàng: 21321


# Xác định top 10 nguyên liệu
- Tạo từ điển `initial_ingredients` để đếm số lượng đơn của từng nguyên liệu.
- Phân tích từng món trong cột `Parsed Items` để ánh xạ thành các nguyên liệu (Chicken, Pizza, Cheese, v.v.) dựa trên từ khóa:
  - Ví dụ: Nếu món có từ "chicken" thì tăng đếm cho "Chicken".
  - Nếu không ánh xạ được, gán vào nhóm "Other".
- Sắp xếp các nguyên liệu theo số lượng đơn giảm dần và chọn top 10.
- Lưu danh sách tên top 10 nguyên liệu vào `top_10_ingredient_names`.
- In danh sách top 10 nguyên liệu cùng số lượng đơn tương ứng.

In [107]:
# Tạo danh sách top 10 nguyên liệu
initial_ingredients = defaultdict(int)
for _, row in df.iterrows():
    for item_name, quantity in row['Parsed Items']:
        ingredients = []
        if 'chicken' in item_name.lower():
            ingredients.append('Chicken')
        if 'pizza' in item_name.lower():
            ingredients.append('Pizza')
        if any(keyword in item_name.lower() for keyword in ['fries', 'french fries']):
            ingredients.append('Fries')
        if 'paneer' in item_name.lower():
            ingredients.append('Paneer')
        if 'rice' in item_name.lower():
            ingredients.append('Rice')
        if any(keyword in item_name.lower() for keyword in ['garlic', 'aioli']):
            ingredients.append('Garlic')
        if any(keyword in item_name.lower() for keyword in ['cheese', 'mozzarella', 'melt']):
            ingredients.append('Cheese')
        if 'pepperoni' in item_name.lower():
            ingredients.append('Pepperoni')
        if any(keyword in item_name.lower() for keyword in ['onion', 'onion rings']):
            ingredients.append('Onion')
        if any(keyword in item_name.lower() for keyword in ['soda', 'ginger ale', 'shikanji', 'iced green tea']):
            ingredients.append('Beverage')
        if any(keyword in item_name.lower() for keyword in ['sauce', 'mayo', 'salsa', 'cafrealsauce', 'dip']):
            ingredients.append('Sauce')
        if 'pide' in item_name.lower():
            ingredients.append('Pide')
        for ing in ingredients:
            initial_ingredients[ing] += quantity
        if not ingredients:
            initial_ingredients['Other'] += quantity

top_10_ingredients = sorted(initial_ingredients.items(), key=lambda x: x[1], reverse=True)[:10]
top_10_ingredient_names = [ing[0] for ing in top_10_ingredients]

print("Top 10 nguyên liệu (1/12/2024 - 31/1/2025):")
for ing, count in top_10_ingredients:
    print(f"  - {ing}: {count} đơn")

Top 10 nguyên liệu (1/12/2024 - 31/1/2025):
  - Chicken: 13243 đơn
  - Pizza: 12653 đơn
  - Cheese: 6391 đơn
  - Garlic: 4153 đơn
  - Paneer: 3827 đơn
  - Pide: 2932 đơn
  - Fries: 2871 đơn
  - Pepperoni: 1829 đơn
  - Other: 1445 đơn
  - Onion: 655 đơn


# Ánh xạ món ăn sang nguyên liệu
- Định nghĩa hàm `map_to_ingredient` để ánh xạ tên món ăn sang danh sách các nguyên liệu:
  - Dựa trên từ khóa (ví dụ: "chicken" → "Chicken", "pizza" → "Pizza").
  - Chỉ ánh xạ các nguyên liệu thuộc `top_10_ingredient_names`.
  - Nếu không ánh xạ được, gán vào nhóm "Other" và đếm số lần món bị gán vào "Other".
- Kiểm tra hàm `map_to_ingredient` với một danh sách món ăn mẫu (ví dụ: "All About Chicken Pizza" → ['Chicken', 'Pizza']).
- In kết quả ánh xạ để xác minh tính đúng đắn của hàm.

In [108]:
# Ánh xạ món ăn sang nhiều nguyên liệu
other_counts = defaultdict(int)
other_counts_last_day = defaultdict(int)

def map_to_ingredient(item_name, date=None):
    ingredients = []
    # Từ khóa cho các nguyên liệu
    ingredient_keywords = {
        'Chicken': ['chicken', 'grilled', 'jamaican', 'peri peri', 'bbq'],
        'Pizza': ['pizza'],
        'Fries': ['fries', 'french fries'],
        'Paneer': ['paneer'],
        'Rice': ['rice'],
        'Garlic': ['garlic', 'aioli'],
        'Cheese': ['cheese', 'mozzarella', 'melt'],
        'Pepperoni': ['pepperoni'],
        'Onion': ['onion', 'onion rings'],
        'Beverage': ['soda', 'ginger ale', 'shikanji', 'iced green tea'],
        'Sauce': ['sauce', 'mayo', 'salsa', 'cafrealsauce', 'dip'],
        'Pide': ['pide']
    }
    
    for ing, keywords in ingredient_keywords.items():
        if ing in top_10_ingredient_names and any(keyword.lower() in item_name.lower() for keyword in keywords):
            ingredients.append(ing)
    
    if not ingredients:
        other_counts[item_name] += 1
        if date and date == '2025-01-31':
            other_counts_last_day[item_name] += 1
        return ['Other']
    return ingredients

# Kiểm tra hàm ánh xạ
test_items = ['All About Chicken Pizza', 'Just Pepperoni Pide', 'Mushroom Mozzarella Melt', 
              'Bone in Jamaican Grilled Chicken', 'Angara Rice', 'Tipsy Tiger Fresh Lime Soda', 
              'Garlic Aioli', 'Onion Rings', 'Bacon Pepperoni Melt', 'Makhani Paneer Pizza', 
              'Chilli Cheese Garlic Bread', 'Indian Salsa', 'Cafreal Sauce', 'french fries']
print("Kiểm tra ánh xạ nguyên liệu:")
for item in test_items:
    print(f"  - {item} → {map_to_ingredient(item)}")

Kiểm tra ánh xạ nguyên liệu:
  - All About Chicken Pizza → ['Chicken', 'Pizza']
  - Just Pepperoni Pide → ['Pepperoni', 'Pide']
  - Mushroom Mozzarella Melt → ['Cheese']
  - Bone in Jamaican Grilled Chicken → ['Chicken']
  - Angara Rice → ['Other']
  - Tipsy Tiger Fresh Lime Soda → ['Other']
  - Garlic Aioli → ['Garlic']
  - Onion Rings → ['Onion']
  - Bacon Pepperoni Melt → ['Cheese', 'Pepperoni']
  - Makhani Paneer Pizza → ['Pizza', 'Paneer']
  - Chilli Cheese Garlic Bread → ['Garlic', 'Cheese']
  - Indian Salsa → ['Other']
  - Cafreal Sauce → ['Other']
  - french fries → ['Fries']


# Lọc và tổng hợp dữ liệu theo ngày
- Lọc dữ liệu từ 1/12/2024 đến 31/1/2025 để phân tích.
- Kiểm tra nếu không có dữ liệu trong khoảng thời gian này, báo lỗi.
- Tổng hợp số lượng nguyên liệu theo ngày:
  - Sử dụng hàm `map_to_ingredient` để ánh xạ các món thành nguyên liệu.
  - Lưu kết quả vào từ điển `ingredient_counts_by_day`.
- Kiểm tra dữ liệu ngày 31/1/2025:
  - In danh sách món ăn ghi nhận (nếu có).
  - In danh sách món bị gán vào "Other" (nếu có).
- Kiểm tra `top_10_ingredient_names` để đảm bảo không rỗng.
- In số ngày có dữ liệu nguyên liệu để xác nhận.

In [109]:
# Lọc dữ liệu từ 1/12/2024 đến 31/1/2025
start_date = pd.to_datetime('2024-12-01')
end_date = pd.to_datetime('2025-01-31')
data = df[(df['Order Placed At'] >= start_date) & (df['Order Placed At'] <= end_date)]

# Kiểm tra dữ liệu
if data.empty:
    raise ValueError("Không có dữ liệu trong khoảng 1/12/2024 - 31/1/2025. Kiểm tra file CSV hoặc cột 'Order Placed At'.")

# Tổng hợp số lượng nguyên liệu theo ngày
ingredient_counts_by_day = defaultdict(lambda: defaultdict(int))
last_day_items = []

for _, row in data.iterrows():
    date = row['Order Placed At'].strftime('%Y-%m-%d')
    for item_name, quantity in row['Parsed Items']:
        ingredients = map_to_ingredient(item_name, date)
        for ingredient in ingredients:
            ingredient_counts_by_day[date][ingredient] += quantity
        if date == '2025-01-31':
            last_day_items.append((item_name, quantity))

# Kiểm tra dữ liệu ngày 31/1/2025
print("\nMón ăn được ghi nhận ngày 31/1/2025:")
if last_day_items:
    for item_name, qty in last_day_items:
        print(f"  - {item_name}: {qty} đơn")
else:
    print("  Không có món nào được ghi nhận. Kiểm tra dữ liệu CSV cho ngày 31/1/2025.")
print("\nMón bị gán vào Other ngày 31/1/2025:")
if other_counts_last_day:
    for item, count in other_counts_last_day.items():
        print(f"  - {item}: {count} lần")
else:
    print("  Không có món nào bị gán vào Other.")

# Kiểm tra top_10_ingredient_names
if not top_10_ingredient_names:
    raise ValueError("Không tìm thấy nguyên liệu nào trong dữ liệu. Kiểm tra cột 'Items in order' hoặc dữ liệu đầu vào.")

print("Số ngày có dữ liệu nguyên liệu:", len(ingredient_counts_by_day))


Món ăn được ghi nhận ngày 31/1/2025:
  Không có món nào được ghi nhận. Kiểm tra dữ liệu CSV cho ngày 31/1/2025.

Món bị gán vào Other ngày 31/1/2025:
  Không có món nào bị gán vào Other.
Số ngày có dữ liệu nguyên liệu: 61


# Tạo chuỗi thời gian
- Tạo DataFrame `ingredient_ts` với chỉ số là các ngày từ 1/12/2024 đến 31/1/2025 và cột là top 10 nguyên liệu, khởi tạo giá trị bằng 0.
- Điền dữ liệu từ `ingredient_counts_by_day` vào `ingredient_ts`:
  - Với mỗi ngày, gán số lượng đơn tương ứng cho từng nguyên liệu.
- Xử lý giá trị 0 bất thường cho ngày 31/1/2025:
  - Tính trung bình 7 ngày trước (24/1/2025 - 30/1/2025) cho từng nguyên liệu.
  - Thay giá trị 0 bằng trung bình này (nếu trung bình lớn hơn 0) và in thông báo.
- In chuỗi thời gian của nguyên liệu "Pizza" trong 14 ngày cuối để kiểm tra.

In [110]:
# Tạo chuỗi thời gian
dates = pd.date_range(start=start_date, end=end_date, freq='D')
ingredient_ts = pd.DataFrame(index=dates, columns=top_10_ingredient_names).fillna(0)

# Điền dữ liệu từ ingredient_counts_by_day
for date in ingredient_counts_by_day:
    if date in ingredient_ts.index.strftime('%Y-%m-%d'):
        for ing in top_10_ingredient_names:
            ingredient_ts.loc[date, ing] = ingredient_counts_by_day[date].get(ing, 0)

# Áp dụng dữ liệu đầu vào cho nguyên liệu chính (giả sử Pizza)


# Xử lý giá trị 0 bất thường cho ngày 31/1/2025
last_date = '2025-01-31'
for ing in top_10_ingredient_names:
    if ingredient_ts.loc[last_date, ing] == 0:
        # Tính trung bình 7 ngày trước (24/1/2025 - 30/1/2025)
        past_7_days = ingredient_ts[ing][-3:-1]
        mean_value = past_7_days.mean() if past_7_days.sum() > 0 else 0
        if mean_value > 0:
            ingredient_ts.loc[last_date, ing] = round(mean_value)
            print(f"Thay giá trị 0 cho {ing} ngày 31/1/2025 bằng trung bình 7 ngày trước: {mean_value:.1f}")

# Kiểm tra dữ liệu
print("\nChuỗi thời gian nguyên liệu (14 ngày cuối, Pizza):")
print(ingredient_ts['Pizza'][-14:])

Thay giá trị 0 cho Chicken ngày 31/1/2025 bằng trung bình 7 ngày trước: 67.0
Thay giá trị 0 cho Pizza ngày 31/1/2025 bằng trung bình 7 ngày trước: 59.5
Thay giá trị 0 cho Cheese ngày 31/1/2025 bằng trung bình 7 ngày trước: 30.0
Thay giá trị 0 cho Garlic ngày 31/1/2025 bằng trung bình 7 ngày trước: 14.5
Thay giá trị 0 cho Paneer ngày 31/1/2025 bằng trung bình 7 ngày trước: 13.5
Thay giá trị 0 cho Pide ngày 31/1/2025 bằng trung bình 7 ngày trước: 11.5
Thay giá trị 0 cho Fries ngày 31/1/2025 bằng trung bình 7 ngày trước: 14.5
Thay giá trị 0 cho Pepperoni ngày 31/1/2025 bằng trung bình 7 ngày trước: 6.5
Thay giá trị 0 cho Other ngày 31/1/2025 bằng trung bình 7 ngày trước: 13.0
Thay giá trị 0 cho Onion ngày 31/1/2025 bằng trung bình 7 ngày trước: 1.5

Chuỗi thời gian nguyên liệu (14 ngày cuối, Pizza):
2025-01-18    102
2025-01-19     66
2025-01-20     27
2025-01-21     48
2025-01-22     64
2025-01-23     52
2025-01-24     72
2025-01-25    194
2025-01-26    106
2025-01-27     37
2025-01-28  

# Kiểm tra mùa vụ
- Định nghĩa hàm `check_seasonality` để kiểm tra tính mùa vụ (chu kỳ tuần) bằng cách tính hàm tự tương quan (ACF):
  - Kiểm tra tại lag 7 và 14, nếu giá trị ACF > 0.15 thì coi là có mùa vụ.
- Kiểm tra mùa vụ cho tất cả top 10 nguyên liệu và in kết quả.
- Vẽ biểu đồ ACF cho nguyên liệu đầu tiên (Chicken) với 13 lag:
  - Lưu biểu đồ vào file `acf_plot.png`.
  - In thông báo xác nhận đã lưu biểu đồ.

In [111]:
# Hàm kiểm tra mùa vụ
def check_seasonality(series):
    try:
        acf_vals = acf(series, nlags=14, fft=False)
        return acf_vals[7] > 0.15 or acf_vals[14] > 0.15
    except:
        return False

# Kiểm tra mùa vụ cho nguyên liệu đầu tiên
has_seasonality = any(check_seasonality(ingredient_ts[ing]) for ing in top_10_ingredient_names)
print(f"Tính mùa vụ (chu kỳ tuần): {has_seasonality}")

# Vẽ biểu đồ ACF cho nguyên liệu đầu tiên
plt.figure()
acf_vals = acf(ingredient_ts[top_10_ingredient_names[0]], nlags=13, fft=False)
plt.stem(range(len(acf_vals)), acf_vals)
plt.title(f'Biểu đồ ACF - {top_10_ingredient_names[0]}')
plt.xlabel('Lag')
plt.ylabel('ACF')
plt.savefig('acf_plot.png')
plt.close()
print("Biểu đồ ACF đã được lưu: 'acf_plot.png'")

Tính mùa vụ (chu kỳ tuần): True
Biểu đồ ACF đã được lưu: 'acf_plot.png'


# Dự báo 7 ngày tới
- Tạo dải ngày dự đoán từ 1/2/2025 đến 7/2/2025.
- Định nghĩa hàm `fit_model` để dự báo chuỗi thời gian:
  - Kiểm tra dữ liệu đầu vào: ít nhất 14 ngày và không quá 30% giá trị 0.
  - Sử dụng SARIMA (nếu có mùa vụ) hoặc ARIMA (nếu không có mùa vụ) từ thư viện `pmdarima`.
  - Nếu dự báo thất bại (NaN hoặc giá trị âm), thay bằng trung bình 7 ngày gần nhất.
- Chọn mô hình (SARIMA/ARIMA) dựa trên kết quả kiểm tra mùa vụ và in thông báo.
- Dự báo số lượng đơn cho từng nguyên liệu trong 7 ngày tới:
  - In dữ liệu đầu vào (14 ngày cuối) và kết quả dự đoán.
- Tổng hợp dự đoán theo ngày, sắp xếp theo số lượng giảm dần.
- In danh sách top 10 nguyên liệu dự đoán cho mỗi ngày từ 1/2/2025 đến 7/2/2025.

In [112]:
# Ngày dự đoán
forecast_dates = pd.date_range(start='2025-02-01', end='2025-02-07', freq='D')
ingredient_forecasts = {}

# Hàm dự đoán
def fit_model(series, steps=7, use_sarima=True):
    try:
        if len(series) < 14:
            raise ValueError("Dữ liệu quá ngắn (<14 ngày)")
        zero_ratio = (series == 0).sum() / len(series)
        if zero_ratio > 0.3:
            raise ValueError(f"Dữ liệu thưa thớt ({zero_ratio*100:.1f}% giá trị 0)")
        
        print(f"Dữ liệu đầu vào (14 ngày cuối): {series[-14:].tolist()}")
        
        if use_sarima:
            model = pm.auto_arima(series, seasonal=True, m=7,
                                  start_p=0, start_q=0, max_p=3, max_q=3,
                                  start_P=0, start_Q=0, max_P=2, max_Q=2,
                                  d=1, D=1, stepwise=True, suppress_warnings=True,
                                  error_action='ignore')
        else:
            model = pm.auto_arima(series, seasonal=False,
                                  start_p=0, start_q=0, max_p=3, max_q=3,
                                  d=1, stepwise=True, suppress_warnings=True,
                                  error_action='ignore')
        forecast = model.predict(n_periods=steps)
        if np.any(np.isnan(forecast)) or np.any(forecast < 0):
            raise ValueError("Dự đoán không hợp lệ (NaN hoặc âm)")
        return np.maximum(forecast.round().astype(int), 0)
    except Exception as e:
        print(f"Lỗi dự đoán cho series: {e}. Sử dụng trung bình 7 ngày gần nhất.")
        mean_value = series[-7:].mean() if series[-7:].sum() > 0 else 0
        print(f"Trung bình 7 ngày gần nhất: {mean_value:.1f}")
        return np.full(steps, round(mean_value)).astype(int)

# Chọn mô hình
model_type = 'SARIMA' if has_seasonality else 'ARIMA'
print(f"Sử dụng mô hình: {model_type}")

# Dự đoán
for ing in top_10_ingredient_names:
    print(f"\nDự đoán cho {ing}:")
    forecast = fit_model(ingredient_ts[ing], steps=7, use_sarima=has_seasonality)
    ingredient_forecasts[ing] = forecast
    print(f"  Kết quả: {forecast.tolist()}")

# Tổng hợp dự đoán
daily_predictions = {}
for idx, date in enumerate(forecast_dates):
    date_str = date.strftime('%Y-%m-%d')
    daily_ingredients = [(ing, ingredient_forecasts[ing][idx]) for ing in top_10_ingredient_names]
    daily_predictions[date_str] = {
        'ingredients': sorted(daily_ingredients, key=lambda x: x[1], reverse=True)
    }

# In dự đoán
print(f"\nDỰ ĐOÁN TOP 10 NGUYÊN LIỆU CHO 7 NGÀY TỚI ({model_type}):")
for date, predictions in daily_predictions.items():
    print(f"\nNgày {date}:")
    print("Top 10 nguyên liệu:")
    for ing, count in predictions['ingredients']:
        print(f"  - {ing}: {count} đơn")

Sử dụng mô hình: SARIMA

Dự đoán cho Chicken:
Dữ liệu đầu vào (14 ngày cuối): [98, 83, 202, 77, 76, 81, 77, 168, 138, 50, 73, 79, 55, 67]


  Kết quả: [107, 86, 130, 58, 60, 54, 56]

Dự đoán cho Pizza:
Dữ liệu đầu vào (14 ngày cuối): [102, 66, 27, 48, 64, 52, 72, 194, 106, 37, 69, 59, 60, 60]
  Kết quả: [123, 76, 28, 68, 52, 50, 61]

Dự đoán cho Cheese:
Dữ liệu đầu vào (14 ngày cuối): [53, 36, 16, 32, 34, 26, 38, 84, 52, 21, 39, 33, 27, 30]
  Kết quả: [64, 40, 11, 29, 27, 20, 26]

Dự đoán cho Garlic:
Dữ liệu đầu vào (14 ngày cuối): [30, 21, 9, 25, 18, 14, 23, 45, 26, 9, 11, 16, 13, 14]
  Kết quả: [35, 22, 8, 15, 14, 10, 15]

Dự đoán cho Paneer:
Dữ liệu đầu vào (14 ngày cuối): [24, 17, 10, 8, 18, 16, 17, 63, 25, 14, 16, 16, 11, 14]
  Kết quả: [25, 14, 7, 21, 12, 12, 14]

Dự đoán cho Pide:
Dữ liệu đầu vào (14 ngày cuối): [13, 15, 9, 7, 6, 12, 11, 53, 21, 10, 16, 13, 10, 12]
  Kết quả: [32, 16, 8, 10, 8, 10, 10]

Dự đoán cho Fries:
Dữ liệu đầu vào (14 ngày cuối): [27, 29, 30, 29, 24, 16, 21, 35, 23, 13, 12, 18, 11, 14]
  Kết quả: [20, 13, 7, 6, 8, 0, 4]

Dự đoán cho Pepperoni:
Dữ liệu đầu vào (14 ngày cuối): [17, 14, 7, 6, 7,

In [117]:
import matplotlib.pyplot as plt

# Vẽ biểu đồ thể hiện sự thay đổi số lượng đơn dự đoán
plt.figure(figsize=(12, 8))

# Sử dụng bảng màu đã định nghĩa trong notebook
for idx, ing in enumerate(top_10_ingredient_names):
    # Lấy dữ liệu dự đoán cho nguyên liệu
    forecast_values = ingredient_forecasts[ing]
    # Vẽ đường cho nguyên liệu
    plt.plot(forecast_dates, forecast_values, label=ing, color=colors[idx % len(colors)], 
             marker='o', linewidth=2, markersize=8)

# Cấu hình biểu đồ
plt.title('Dự đoán thay đổi số lượng đơn nguyên liệu (1/2/2025 - 7/2/2025)', fontsize=14, pad=10)
plt.xlabel('Ngày', fontsize=12)
plt.ylabel('Số lượng đơn', fontsize=12)
plt.legend(title='Nguyên liệu', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()

# Lưu biểu đồ
plt.savefig('ingredient_forecast_trend.png', bbox_inches='tight')
plt.close()

print("\nBiểu đồ thay đổi số lượng đơn dự đoán đã được lưu: 'ingredient_forecast_trend.png'")


Biểu đồ thay đổi số lượng đơn dự đoán đã được lưu: 'ingredient_forecast_trend.png'


In [118]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Dữ liệu thực tế 14 ngày trước (18/1/2025 - 31/1/2025)
historical_start = pd.to_datetime('2025-01-18')
historical_end = pd.to_datetime('2025-01-31')
historical_dates = pd.date_range(start=historical_start, end=historical_end, freq='D')
historical_data = ingredient_ts.loc[historical_start:historical_end]

# In dữ liệu thực tế và dự đoán để so sánh
print("\nSO SÁNH DỮ LIỆU THỰC TẾ (18/1/2025 - 31/1/2025) VỚI DỰ ĐOÁN (1/2/2025 - 7/2/2025):\n")

for ing in top_10_ingredient_names:
    print(f"\nNguyên liệu: {ing}")
    print("Dữ liệu thực tế (14 ngày, 18/1/2025 - 31/1/2025):")
    print(historical_data[ing].values.tolist())
    print(f"Trung bình thực tế: {historical_data[ing].mean():.1f} đơn")
    print("Dữ liệu dự đoán (7 ngày, 1/2/2025 - 7/2/2025):")
    print(ingredient_forecasts[ing].tolist())
    print(f"Trung bình dự đoán: {np.mean(ingredient_forecasts[ing]):.1f} đơn\n")

# Vẽ biểu đồ kết hợp dữ liệu thực tế và dự đoán
plt.figure(figsize=(14, 8))

for idx, ing in enumerate(top_10_ingredient_names):
    # Dữ liệu thực tế
    historical_values = historical_data[ing].values
    # Dữ liệu dự đoán
    forecast_values = ingredient_forecasts[ing]
    
    # Vẽ đường thực tế (liên tục)
    plt.plot(historical_dates, historical_values, label=f'{ing} (Thực tế)', 
             color=colors[idx % len(colors)], linewidth=2, marker='o')
    # Vẽ đường dự đoán (nét đứt)
    plt.plot(forecast_dates, forecast_values, label=f'{ing} (Dự đoán)', 
             color=colors[idx % len(colors)], linestyle='--', linewidth=2, marker='x')

# Cấu hình biểu đồ
plt.title('So sánh dữ liệu thực tế (18/1/2025 - 31/1/2025) và dự đoán (1/2/2025 - 7/2/2025)', 
          fontsize=14, pad=10)
plt.xlabel('Ngày', fontsize=12)
plt.ylabel('Số lượng đơn', fontsize=12)
plt.legend(title='Nguyên liệu', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()

# Lưu biểu đồ
plt.savefig('historical_vs_forecast_trend.png', bbox_inches='tight')
plt.close()

print("\nBiểu đồ so sánh dữ liệu thực tế và dự đoán đã được lưu: 'historical_vs_forecast_trend.png'")


SO SÁNH DỮ LIỆU THỰC TẾ (18/1/2025 - 31/1/2025) VỚI DỰ ĐOÁN (1/2/2025 - 7/2/2025):


Nguyên liệu: Chicken
Dữ liệu thực tế (14 ngày, 18/1/2025 - 31/1/2025):
[98, 83, 202, 77, 76, 81, 77, 168, 138, 50, 73, 79, 55, 67]
Trung bình thực tế: 94.6 đơn
Dữ liệu dự đoán (7 ngày, 1/2/2025 - 7/2/2025):
[107, 86, 130, 58, 60, 54, 56]
Trung bình dự đoán: 78.7 đơn


Nguyên liệu: Pizza
Dữ liệu thực tế (14 ngày, 18/1/2025 - 31/1/2025):
[102, 66, 27, 48, 64, 52, 72, 194, 106, 37, 69, 59, 60, 60]
Trung bình thực tế: 72.6 đơn
Dữ liệu dự đoán (7 ngày, 1/2/2025 - 7/2/2025):
[123, 76, 28, 68, 52, 50, 61]
Trung bình dự đoán: 65.4 đơn


Nguyên liệu: Cheese
Dữ liệu thực tế (14 ngày, 18/1/2025 - 31/1/2025):
[53, 36, 16, 32, 34, 26, 38, 84, 52, 21, 39, 33, 27, 30]
Trung bình thực tế: 37.2 đơn
Dữ liệu dự đoán (7 ngày, 1/2/2025 - 7/2/2025):
[64, 40, 11, 29, 27, 20, 26]
Trung bình dự đoán: 31.0 đơn


Nguyên liệu: Garlic
Dữ liệu thực tế (14 ngày, 18/1/2025 - 31/1/2025):
[30, 21, 9, 25, 18, 14, 23, 45, 26, 9, 11, 16,

# So sánh với dữ liệu thực tế 14 ngày trước và vẽ biểu đồ liên tục
- Trích xuất dữ liệu thực tế từ 18/1/2025 đến 31/1/2025 từ `ingredient_ts`.
- In dữ liệu thực tế và dự đoán để so sánh:
  - Dữ liệu thực tế (14 ngày) và trung bình thực tế.
  - Dữ liệu dự đoán (7 ngày) và trung bình dự đoán.
- Tạo dải ngày liên tục từ 18/1/2025 đến 7/2/2025.
- Vẽ biểu đồ kết hợp dữ liệu thực tế và dự đoán:
  - Kết hợp dữ liệu thực tế và dự đoán thành một mảng duy nhất để vẽ đường liên tục.
  - Vẽ lại đoạn dự đoán (1/2/2025 - 7/2/2025) với kiểu đường nét đứt để phân biệt.
  - Sử dụng bảng màu `colors` để giữ màu nhất quán cho từng nguyên liệu.
- Cấu hình biểu đồ với tiêu đề, nhãn trục, chú thích và lưới.
- Lưu biểu đồ vào file `continuous_historical_vs_forecast_trend.png`.
- In thông báo xác nhận đã lưu biểu đồ.

In [120]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Dữ liệu thực tế 14 ngày trước (18/1/2025 - 31/1/2025)
historical_start = pd.to_datetime('2025-01-18')
historical_end = pd.to_datetime('2025-01-31')
historical_dates = pd.date_range(start=historical_start, end=historical_end, freq='D')
historical_data = ingredient_ts.loc[historical_start:historical_end]

# Dải ngày liên tục từ 18/1/2025 đến 7/2/2025
full_dates = pd.date_range(start='2025-01-18', end='2025-02-07', freq='D')

# Vẽ biểu đồ
plt.figure(figsize=(14, 8))

for idx, ing in enumerate(top_10_ingredient_names):
    # Kết hợp dữ liệu thực tế và dự đoán thành một mảng duy nhất
    historical_values = historical_data[ing].values
    forecast_values = ingredient_forecasts[ing]
    # Tạo mảng đầy đủ: dữ liệu thực tế + dự đoán
    full_values = np.concatenate([historical_values, forecast_values])
    
    # Vẽ đường liên tục cho toàn bộ dữ liệu
    plt.plot(full_dates, full_values, label=ing, color=colors[idx % len(colors)], 
             linewidth=2, marker='o')
    
    # Vẽ lại đoạn dự đoán (1/2/2025 - 7/2/2025) với kiểu nét đứt
    plt.plot(forecast_dates, forecast_values, color=colors[idx % len(colors)], 
             linestyle='--', linewidth=2, marker='x')

# Cấu hình biểu đồ
plt.title('So sánh dữ liệu thực tế (18/1/2025 - 31/1/2025) và dự đoán (1/2/2025 - 7/2/2025)', 
          fontsize=14, pad=10)
plt.xlabel('Ngày', fontsize=12)
plt.ylabel('Số lượng đơn', fontsize=12)
plt.legend(title='Nguyên liệu', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()

# Lưu biểu đồ
plt.savefig('continuous_historical_vs_forecast_trend.png', bbox_inches='tight')
plt.close()

print("\nBiểu đồ liên tục so sánh dữ liệu thực tế và dự đoán đã được lưu: 'continuous_historical_vs_forecast_trend.png'")


Biểu đồ liên tục so sánh dữ liệu thực tế và dự đoán đã được lưu: 'continuous_historical_vs_forecast_trend.png'


In [114]:
from prophet import Prophet

def fit_prophet(series, dates, steps=7):
    df = pd.DataFrame({'ds': dates, 'y': series})
    model = Prophet(daily_seasonality=False, weekly_seasonality=True, yearly_seasonality=False)
    model.fit(df)
    future = model.make_future_dataframe(periods=steps, freq='D')
    forecast = model.predict(future)
    return forecast['yhat'][-steps:].values.round().astype(int)

# Thử Prophet cho Chicken
chicken_series = train_data['Chicken']
chicken_dates = train_data.index
prophet_forecast = fit_prophet(chicken_series, chicken_dates, steps=7)
print("\nDự đoán Prophet cho Chicken (25/1/2025 - 31/1/2025):")
print(prophet_forecast)

Importing plotly failed. Interactive plots will not work.
17:11:45 - cmdstanpy - INFO - Chain [1] start processing
17:11:45 - cmdstanpy - INFO - Chain [1] done processing



Dự đoán Prophet cho Chicken (25/1/2025 - 31/1/2025):
[109  82  78  81  88  80  98]
